# What is data governance? A data engineer's version

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/governance-sample/what_is_data_governance.ipynb)

| Field | Details |
| :--- | :--- |
| **Audience** | Data Engineers, Analytics Engineers, Platform Engineers |
| **Level** | Level 100 — Foundational Engineering Walkthrough |
| **Prerequisites** | Google Cloud Project with BigQuery, Cloud Storage, and Knowledge Catalog APIs enabled |
| **Estimated Time** | 10 minutes |

### Learning objectives

By the end of this walkthrough, you will be able to:
- Discover **BigQuery** tables and **Cloud Storage** files with a single **Knowledge Catalog** search query instead of asking around in Slack.
- Trace a broken downstream dashboard table back to its upstream source table and inspect the SQL transformation that feeds it.
- Run a 20-line Python snippet to fetch a table's update timestamp and on-call owner contact directly from **Knowledge Catalog**.

---

## 1. The 2 AM silent data drop

It is two in the morning, and your phone rings. The executive sales dashboard is wrong—revenue dropped by half overnight, and the leadership meeting starts at nine.

You open your pipeline orchestration dashboard. **Every single job is green.** No errors, no failed tasks, nothing crashed.

Yesterday, an upstream application team stopped sending one specific checkout event type. The table schema did not change, and your SQL transformation ran without a single syntax or runtime error. It simply processed half the rows quietly. Nothing broke; something quietly went missing.

### Data swamp vs. data with context

When most engineers hear "data governance," they picture a 50-page policy PDF that nobody reads and meetings nobody wants to attend. From an engineering perspective, governance is not bureaucracy—it is how your data platform answers four operational questions before 9:00 AM:

1. **Discovery**: What data do we have across BigQuery and Cloud Storage, and where does it live?
2. **Quality**: Can I trust the numbers inside this table right now?
3. **Lineage**: Where did this number come from, and which upstream table caused the drop?
4. **Access & Ownership**: Who owns that upstream pipeline at 2 AM, and who is authorized to read sensitive columns?

In a **data swamp**, your storage works and your compute is fast, but nobody knows what anything means. If you see three tables named `customer_orders_final` and have to ask five teams in Slack which one is authoritative, you do not have governance.

When you govern that same data with **Knowledge Catalog**, you do not move a single byte of data or rewrite your pipelines. You attach operational context—unified search indexes, upstream lineage graphs, and owner contacts—directly onto the tables and files you already have.

> ℹ️ **Scope Boundary (Level 100)**: In this walkthrough, we focus hands-on on **Discovery**, **Lineage**, and **Ownership** so you can trace a broken dashboard and page the right owner in minutes. Automated **Data Quality** scans and bulk metadata tagging across hundreds of tables are covered in the next episode of this series.

## 2. Configure your Google Cloud project

Authenticate your Google Colab session and set your `PROJECT_ID` below. By default, the walkthrough uses a sample BigQuery dataset named `sales_mart` in `us-central1` and a regional Cloud Storage bucket.

In [ ]:
# @title Project configuration
import os
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

DATASET_ID = "sales_mart"
OWNER_EMAIL = "data-eng-oncall@example.com"
BUCKET_NAME = f"{PROJECT_ID}-sales-archive"

if not PROJECT_ID or PROJECT_ID in {"your-project-id", "YOUR-PROJECT-ID", "YOUR_PROJECT_ID"}:
    raise ValueError("Please set PROJECT_ID to your active Google Cloud project before running.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print(f"✓ Configured project '{PROJECT_ID}' (location: {LOCATION}, dataset: {DATASET_ID})")

## 3. Install SDK clients & bootstrap the demo environment

Running the cell below installs the Google Cloud client libraries and provisions the 2 AM scenario directly in your project:
- **`sales_mart.customer_orders`**: Upstream BigQuery table where `2026-09-16` order volume and revenue (`2` orders, `$500.00`) drop by **50%** compared to `2026-09-15` (`4` orders, `$1,000.00`).
- **`customer_orders_archive.csv`**: Historical CSV archive in Cloud Storage registered as an External Table (`sales_mart.customer_orders_archive`) so **Knowledge Catalog** indexes it automatically.
- **`sales_mart.daily_revenue`**: Downstream executive dashboard table built via `CREATE OR REPLACE TABLE ... AS SELECT` (which triggers automatic BigQuery lineage capture) and tagged with an on-call owner contact (`data-eng-oncall@example.com`).

In [ ]:
!pip install -q google-cloud-bigquery google-cloud-storage google-cloud-dataplex google-cloud-datacatalog-lineage

import importlib
import time

importlib.invalidate_caches()

from google.api_core.exceptions import AlreadyExists, NotFound
from google.cloud import bigquery, dataplex_v1, storage
from google.cloud import datacatalog_lineage_v1 as lineage_v1
from google.protobuf import field_mask_pb2, struct_pb2

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()
lineage_client = lineage_v1.LineageClient()

# 1. Create the sales_mart dataset in BigQuery
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
bq_client.create_dataset(dataset_ref, exists_ok=True)

# 2. Upload customer_orders_archive.csv to Cloud Storage & register an External Table
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(bucket, location=LOCATION)

bucket.blob("customer_orders_archive.csv").upload_from_string(
    "order_id,customer_id,order_timestamp,order_amount,status\n"
    "ORD-0901,CUST-01,2026-09-14 10:00:00,250.00,COMPLETED\n"
    "ORD-0902,CUST-02,2026-09-14 14:30:00,250.00,COMPLETED\n",
    content_type="text/csv",
)

bq_client.query(f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.customer_orders_archive` (
    order_id STRING, customer_id STRING, order_timestamp TIMESTAMP, order_amount NUMERIC, status STRING
)
OPTIONS (format = 'CSV', skip_leading_rows = 1, uris = ['gs://{BUCKET_NAME}/customer_orders_archive.csv'])
""").result()

# 3. Create upstream table sales_mart.customer_orders (2026-09-15: 4 orders / $1,000 -> 2026-09-16: 2 orders / $500)
bq_client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.customer_orders` AS
SELECT 'ORD-1001' AS order_id, 'CUST-10' AS customer_id, TIMESTAMP '2026-09-15 09:15:00+00' AS order_timestamp, NUMERIC '250.00' AS order_amount, 'COMPLETED' AS status UNION ALL
SELECT 'ORD-1002', 'CUST-11', TIMESTAMP '2026-09-15 11:30:00+00', NUMERIC '250.00', 'COMPLETED' UNION ALL
SELECT 'ORD-1003', 'CUST-12', TIMESTAMP '2026-09-15 15:45:00+00', NUMERIC '250.00', 'COMPLETED' UNION ALL
SELECT 'ORD-1004', 'CUST-13', TIMESTAMP '2026-09-15 18:20:00+00', NUMERIC '250.00', 'COMPLETED' UNION ALL
SELECT 'ORD-2001', 'CUST-21', TIMESTAMP '2026-09-16 10:05:00+00', NUMERIC '250.00', 'COMPLETED' UNION ALL
SELECT 'ORD-2002', 'CUST-22', TIMESTAMP '2026-09-16 16:40:00+00', NUMERIC '250.00', 'COMPLETED'
""").result()

# 4. Create downstream table sales_mart.daily_revenue via SQL CTAS so BigQuery records lineage automatically
bq_client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.daily_revenue` AS
SELECT
    DATE(order_timestamp) AS revenue_date,
    COUNT(*) AS order_count,
    SUM(order_amount) AS total_revenue
FROM `{PROJECT_ID}.{DATASET_ID}.customer_orders`
GROUP BY revenue_date
""").result()

# 5. Attach on-call owner contact Aspect (table-contacts) onto sales_mart.daily_revenue
ASPECT_TYPE_ID = "table-contacts"
aspect_type_path = f"projects/{PROJECT_ID}/locations/{LOCATION}/aspectTypes/{ASPECT_TYPE_ID}"
try:
    catalog_client.create_aspect_type(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}",
        aspect_type_id=ASPECT_TYPE_ID,
        aspect_type=dataplex_v1.AspectType(
            description="On-call owner contact metadata.",
            metadata_template=dataplex_v1.AspectType.MetadataTemplate(
                name="TableContacts",
                type="record",
                record_fields=[
                    dataplex_v1.AspectType.MetadataTemplate(
                        name="owner_email",
                        type="string",
                        index=1,
                        constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(required=True),
                    ),
                    dataplex_v1.AspectType.MetadataTemplate(name="slack_channel", type="string", index=2),
                ],
            ),
        ),
    ).result()
except AlreadyExists:
    print(f"AspectType already exists: {ASPECT_TYPE_ID}")

entry_name = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/"
    f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/daily_revenue"
)
aspect_key = f"{PROJECT_ID}.{LOCATION}.{ASPECT_TYPE_ID}"
contact_payload = struct_pb2.Struct()
contact_payload.update({"owner_email": OWNER_EMAIL, "slack_channel": "#data-eng-oncall"})

for attempt in range(12):
    try:
        catalog_client.update_entry(
            request=dataplex_v1.UpdateEntryRequest(
                entry=dataplex_v1.Entry(
                    name=entry_name,
                    aspects={aspect_key: dataplex_v1.Aspect(aspect_type=aspect_type_path, data=contact_payload)},
                ),
                update_mask=field_mask_pb2.FieldMask(paths=["aspects"]),
                aspect_keys=[aspect_key],
            )
        )
        break
    except NotFound:
        time.sleep(5)
else:
    raise RuntimeError(f"BigQuery entry did not sync to Knowledge Catalog within 60s: {entry_name}")

print("✓ Demo environment ready. Daily revenue rows in sales_mart.daily_revenue:")
for row in bq_client.query(f"SELECT revenue_date, order_count, total_revenue FROM `{PROJECT_ID}.{DATASET_ID}.daily_revenue` ORDER BY revenue_date").result():
    print(f"  {row.revenue_date}: {row.order_count} orders | ${row.total_revenue:.2f}")

## 4. Discovery: search across BigQuery and Cloud Storage

When a dashboard breaks, your first question is simple: *where does our customer order data actually live?*

In most production stacks, relational tables live in **BigQuery** while historical exports sit as CSV or Parquet files in **Cloud Storage**. Because **Knowledge Catalog** indexes BigQuery tables and external Cloud Storage filesets automatically, one query surfaces both assets immediately.

> ℹ️ **Note (Catalog Indexing Latency & Table Recreation)**: After creating new tables in BigQuery, **Knowledge Catalog** typically takes **1 to 3 minutes** to synchronize the metadata into the search index. If you delete and recreate the `sales_mart` dataset repeatedly during testing, the search index can briefly point to the deleted revision (`Failed to load`) until re-indexing completes.

### How to search in the Knowledge Catalog console UI
1. Open **[Knowledge Catalog Search in the Google Cloud Console](https://console.cloud.google.com/dataplex/dp-search-nl)**.
2. Select your Google Cloud project in the top bar, type `customer_orders` (or `daily_revenue`) in the search box, and press **Enter**.
3. Inspect the returned entries (`sales_mart.customer_orders`, `sales_mart.customer_orders_archive`, and `sales_mart.daily_revenue`) along with their column schemas (`order_amount`, `total_revenue`) in a single unified view.

### Search via the Python SDK
You can run the exact same search from Python using `catalog_client.search_entries`:

In [ ]:
search_results = list(
    catalog_client.search_entries(
        request=dataplex_v1.SearchEntriesRequest(
            name=f"projects/{PROJECT_ID}/locations/global",
            query="name:customer_orders",
            scope=f"projects/{PROJECT_ID}",
            page_size=20,
        )
    )
)

discovered_rows = []
for result_item in search_results:
    entry = result_item.dataplex_entry
    system_name = entry.entry_source.system or "BigQuery"
    entry_id = entry.name.split("/")[-1]
    fqn = entry.fully_qualified_name or entry.name
    discovered_rows.append((system_name, entry_id, fqn))

assert len(discovered_rows) >= 2, f"Expected both native and Cloud Storage-backed assets, found: {discovered_rows}"

print(f"{'Storage System':<16} | {'Catalog Entry ID':<34} | {'Fully Qualified Name'}")
print("-" * 95)
for system_name, entry_id, fqn in discovered_rows:
    print(f"{system_name:<16} | {entry_id:<34} | {fqn}")

## 5. Lineage: trace the 50% revenue drop upstream

Now that we know what tables exist, we return to our 2 AM mystery: **why did `sales_mart.daily_revenue` drop by 50% when every pipeline job succeeded?**

### How to inspect visual lineage in the Knowledge Catalog console UI
1. Open **[Knowledge Catalog Search in the Google Cloud Console](https://console.cloud.google.com/dataplex/dp-search-nl)** and click the **`daily_revenue`** entry.
2. Select the **Lineage** tab to view the visual graph connecting upstream **`sales_mart.customer_orders`** to downstream **`sales_mart.daily_revenue`**.
3. Click the transformation process node between the two tables to inspect the exact `CREATE OR REPLACE TABLE ... AS SELECT` SQL statement that produced the table.

### Query upstream lineage via the Python SDK
We can query that same lineage graph from Python using `LineageClient`:

> ℹ️ **Note**: Automatic BigQuery query lineage is indexed asynchronously in the background and typically takes **15 to 30 minutes** after the `CREATE OR REPLACE TABLE ... AS SELECT` query in Section 3 completes to appear in `lineage_client.search_links()`.

In [ ]:
target_table = lineage_v1.EntityReference(
    fully_qualified_name=f"bigquery:{PROJECT_ID}.{DATASET_ID}.daily_revenue"
)

# Query both the regional endpoint (e.g., us-central1) and the 'us' multi-region
# (matching how the Knowledge Catalog Console UI fans out SearchLinks requests)
upstream_links = []
seen_edges = set()
for loc in dict.fromkeys([LOCATION.lower(), "us"]):
    for link in lineage_client.search_links(
        request=lineage_v1.SearchLinksRequest(
            parent=f"projects/{PROJECT_ID}/locations/{loc}",
            target=target_table,
        )
    ):
        edge = (link.source.fully_qualified_name, link.target.fully_qualified_name)
        if edge not in seen_edges:
            seen_edges.add(edge)
            upstream_links.append((loc, link))

if upstream_links:
    for loc, link in upstream_links:
        print(f"Upstream Source : {link.source.fully_qualified_name}")
        print(f"Downstream Table: {link.target.fully_qualified_name}")
        for proc_link in lineage_client.batch_search_link_processes(
            request=lineage_v1.BatchSearchLinkProcessesRequest(
                parent=f"projects/{PROJECT_ID}/locations/{loc}",
                links=[link.name],
            )
        ):
            proc = lineage_client.get_process(name=proc_link.process)
            print(f"Process         : {proc.display_name} -> {dict(proc.attributes)}")
else:
    print(
        f"No lineage links returned yet for {target_table.fully_qualified_name}.\n"
        "BigQuery publishes CREATE OR REPLACE TABLE ... AS SELECT lineage asynchronously (15-30 minutes).\n"
        "Re-run this cell or check the Lineage tab in https://console.cloud.google.com/dataplex/dp-search-nl once indexing completes."
    )

## 6. Ownership: fetch the on-call contact in 20 lines of Python

Tracing `daily_revenue` back to `customer_orders` tells us *where* the drop happened, but at two in the morning we still need to know *who* to contact.

While schema and lineage are captured automatically, table ownership is human context: someone on the team fills in the **Contacts** metadata once so nobody has to guess during an outage.

### How to inspect ownership metadata in the Knowledge Catalog console UI
1. Open **[Knowledge Catalog Search in the Google Cloud Console](https://console.cloud.google.com/dataplex/dp-search-nl)** and click **`daily_revenue`**.
2. In the **Overview** / **Details** tab under **Aspects**, inspect the attached `table-contacts` aspect showing `owner_email` (`data-eng-oncall@example.com`) and `slack_channel` (`#data-eng-oncall`).

### Fetch table context in 20 lines of Python
As engineers, we do not want to click through a web UI every time an alert fires. The cell below fetches the `daily_revenue` catalog entry with `EntryView.ALL` and prints its update timestamp alongside the attached `table-contacts` aspect.

In [ ]:
entry_name = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/"
    f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/daily_revenue"
)

entry = catalog_client.get_entry(
    request={"name": entry_name, "view": dataplex_v1.EntryView.ALL}
)

print(f"Table FQN : {entry.fully_qualified_name}")
print(f"Updated   : {entry.update_time}")

# Knowledge Catalog keys aspects by '<PROJECT_NUMBER>.<LOCATION>.table-contacts'
matched_key, contacts_aspect = next(
    ((k, v) for k, v in entry.aspects.items() if k.endswith(f".{LOCATION}.table-contacts")),
    (None, None),
)
assert contacts_aspect is not None, f"Expected aspect '*.{LOCATION}.table-contacts' on {entry.name}"

owner_contacts = dict(contacts_aspect.data)
print(f"Aspect [{matched_key}] -> {owner_contacts}")

## 7. Clean up resources

Run the cell below to remove the sample BigQuery dataset, Cloud Storage archive bucket, and custom AspectType created in Section 3.

In [ ]:
from google.api_core.exceptions import NotFound

if "bq_client" in locals():
    bq_client.delete_dataset(f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True)
    print(f"Deleted BigQuery dataset: {PROJECT_ID}.{DATASET_ID}")

if "storage_client" in locals():
    try:
        storage_client.bucket(BUCKET_NAME).delete(force=True)
        print(f"Deleted Cloud Storage bucket: gs://{BUCKET_NAME}")
    except NotFound as exc:
        print(f"Already removed: {exc}")

if "catalog_client" in locals():
    try:
        catalog_client.delete_aspect_type(name=aspect_type_path).result()
        print(f"Deleted Knowledge Catalog AspectType: {aspect_type_path}")
    except NotFound as exc:
        print(f"Already removed: {exc}")